In [4]:
def showLlista(L):
    for i in range(len(L)):
        show(L[i])
        
def transformaEnLlista(L):
    if not (isinstance(L, list)):
        return [L]
    return L

In [5]:
# Una funció per generalitzar la variable a escollir, una heurística
def grausPerVariable(polinomis, variables):
    # Cada element correspon a una llista de una variable, i en cada element d'aquesta correspon 
    # al grau d'aquesta variable en la expressió
    graus=[] 
    for v in variables:
        graus.append([p.degree(v) for p in polinomis])
    
    return graus

In [6]:
def heuristicaResultants(polinomis, variables):
    graus = grausPerVariable(polinomis, variables)
    
    llistaMidaSylvester = []

    for L in graus:
        indexs_valids = [i for i in range(len(L)) if L[i] > 0]

        if len(indexs_valids) < 2:
            llistaMidaSylvester.append(10000)
            continue
        
        minim = min([L[i] for i in indexs_valids])
        idx_minim = L.index(minim)
        
        midaSylvesterMax = 0

        for i in range(len(L)):
            if i == idx_minim or L[i] == 0:
                continue

            if minim + L[i] > midaSylvesterMax:
                midaSylvesterMax = minim + L[i]
                    
        llistaMidaSylvester.append(midaSylvesterMax)
    
    idx_millorVariable = llistaMidaSylvester.index(min(llistaMidaSylvester))

    L = graus[idx_millorVariable]
    indexs_valids = [i for i in range(len(L)) if L[i] > 0]
    minim = min([L[i] for i in indexs_valids])
    idx_polinomiPivot = L.index(minim)
    
#     print(f'La millor variable és {variables[idx_millorVariable]}')
#     print(f'Pivot {idx_polinomiPivot}     {polinomis[idx_polinomiPivot]}')
#     print(f'graus {graus[idx_millorVariable]}\n\n')

    return idx_millorVariable, idx_polinomiPivot

In [7]:
def esZero(arrel, tol=1e-10):
    arrel = arrel.simplify_full()
    # solve sol treballar amb expressions simbóliques (quan es tracta de complexos no sol ser així)
    if (arrel == 0):
        return True
    # comprovació numèrica adicional
    if (abs(CC(arrel)) < tol):
        return True
    
    return False

In [8]:
def auxSolve(equacions, variables):
    # to_poly_solve (per trobar també les complexes) pot no funcionar quan es pasa més d'una variable
    try:
        return solve(equacions, variables, to_poly_solve=True)
    except:
        pass
        
    return solve(equacions, variables[0])

In [9]:
def resolPerResultants(polinomis, variables, tempsAturada = 120):
    if (len(polinomis) == 1 or len(variables) == 1):
        return auxSolve([p == 0 for p in polinomis], variables)

    # Escollim la millor variable i el polinomi pivot
    idx_variable, idx_pivot = heuristicaResultants(polinomis, variables)
    variable_eliminada = variables[idx_variable]
    polinomi_pivot = polinomis[idx_pivot]

    nou_sistema = [] # amb resultants

    for j in range(len(polinomis)):
        if (j == idx_pivot):
            continue
        
        try:
            alarm(tempsAturada)
            resultant = polinomi_pivot.resultant(polinomis[j], variable_eliminada)
        except AlarmInterrupt:
            print("S'ha excedit el temps de càlcul del resultant.")
            resultant = None
        finally:
            cancel_alarm()
            
        if (resultant is None):
            return None
            
        resultant = resultant.expand()
    
        if (resultant == 0):
            # En aquest cas ens indica que existeix una arrel comuna per als dos polinomis que conformen el resultant, però
            # aquí no la podem trobar. No dona informació nova.
            continue
        if (len(resultant.variables()) == 0):
            print("El sistema no té solució, s'ha trobat la igualtat que una constant no nula és igual a 0")
            return []
        
        coeficients = [c[0] for c in resultant.coefficients()]
        c = gcd(coeficients)
        resultant /= c
        resultant = resultant.simplify_full()
        nou_sistema.append(resultant)

        
    if (len(nou_sistema) == 0):
        return []

    noves_variables = variables[:]
    noves_variables.pop(idx_variable)

    # Resolem recursivament el nou nivell
    solucions_reduides = resolPerResultants(nou_sistema, noves_variables)
    if (solucions_reduides is None):
        return None

    # Reconstruir la variable eliminada
    solucions_completes = []
    for sol in solucions_reduides:
        sol = transformaEnLlista(sol)
        polinomis_substituits = [p.subs(sol).simplify_full() for p in polinomis]
        
        if all(p==0 for p in polinomis_substituits):
            #si tots són 0, la variable eliminada queda lliure, pero si no es fa res, queda solucions = []
            r = var(f'r_{variable_eliminada}')
            sols_variable = [[variable_eliminada == r]]
        else: 
            sols_variable = auxSolve([p == 0 for p in polinomis_substituits], [variable_eliminada])
            
        for s in sols_variable:
            s = transformaEnLlista(s)
            sol_completa = sol + s

            # Verificació final en el sistema original d'aquest nivell
            if all(esZero(p.subs(sol_completa)) for p in polinomis):
                solucions_completes.append(sol_completa)
    
    return solucions_completes

## Sistemes inclosos al treball

In [7]:
var('x y z t u')

(x, y, z, t, u)

In [8]:
pols = [
    x^2*y + x*y^2 + x^2 + y^2 - 1,
    x^3 + y^3 - 4*x*y + 1
]
variables = [x, y]
resolPerResultants(pols, variables)
#retorna només les solucions reals

[[y == -1, x == 0], [y == 0, x == -1]]

In [9]:
pols = [
    z^2 - 1,
    z - x,
    z - y
]
variables = [x, y, z]
resolPerResultants(pols, variables)

[[y == -1, x == -1, z == -1], [y == 1, x == 1, z == 1]]

In [10]:
var('a0 a1 a2')

P = a0 + a1*x + a2*x^2 + x^3
P1 = diff(P,x)
P2 = diff(P,x,2)

pols = [
    P.resultant(P1,x),
    P.resultant(P2,x)
]
variables = [a0,a1,a2]
resolPerResultants(pols, variables)

[[a1 == 1/3*a2^2, a0 == 1/27*a2^3]]

In [11]:
var('a0 a1 a2')

P = a0 + a1*x + a2*x^2 + x^4
P1 = diff(P,x)
P2 = diff(P,x,2)
P3 = diff(P,x,3)

pols = [
    P.resultant(P1,x),
    P.resultant(P2,x),
    P.resultant(P3,x)
]
variables = [a0,a1,a2]
resolPerResultants(pols, variables)

[[a2 == 0, a1 == 0, a0 == 0]]

In [12]:
var('a0 a1 a2 a3')

P = a0 + a1*x + a2*x^2 + a3*x^3 + x^5
P1 = diff(P,x)
P2 = diff(P,x,2)
P3 = diff(P,x,3)
P4 = diff(P,x,4)

pols = [
    P.resultant(P1,x),
    P.resultant(P2,x),
    P.resultant(P3,x),
    P.resultant(P4,x)
]
variables = [a0,a1,a2,a3]
resolPerResultants(pols, variables)

[[a3 == 0, a1 == 0, a2 == 0, a0 == 0]]

In [13]:
#Grau n=5: força bruta
var('a0 a1 a2 a3 a4 x')

P = a0 + a1*x + a2*x^2 + a3*x^3 + a4*x^4 + x^5
P1 = diff(P,x)
P2 = diff(P,x,2)
P3 = diff(P,x,3)
P4 = diff(P,x,4)

pols = [
    P.resultant(P1,x),
    P.resultant(P2,x),
    P.resultant(P3,x),
    P.resultant(P4,x)
]

variables = [a0,a1,a2,a3,a4]

resolPerResultants(pols, variables)

[[a3 == 2/5*a4^2, a2 == 2/25*a4^3, a1 == 1/125*a4^4, a0 == 1/3125*a4^5]]

In [14]:
# var('a0 a1 a2 a3 a4 x')

# P = a0 + a1*x + a2*x^2 + a3*x^3 + a4*x^4 + x^6
# P1 = diff(P,x)
# P2 = diff(P,x,2)
# P3 = diff(P,x,3)
# P4 = diff(P,x,4)
# P5 = diff(P,x,5)

# pols = [
#     P.resultant(P1,x),
#     P.resultant(P2,x),
#     P.resultant(P3,x),
#     P.resultant(P4,x),
#     P.resultant(P5,x)
# ]

# variables = [a0,a1,a2,a3,a4]

# resolPerResultants(pols, variables)

# Testing
Exemples generats per inteligència artificial per fer un testing bàsic.

In [15]:
pols = [
    x + y - 3,
    x - y - 1
]
variables = [x,y]
resolPerResultants(pols, variables)

[[y == 1, x == 2]]

In [16]:
pols = [
    x^2 - 1,
    y - x
]
variables = [x,y]
resolPerResultants(pols, variables)

[[y == -1, x == -1], [y == 1, x == 1]]

In [17]:
pols = [
    x^2 - 1,
    y + x
]
variables = [x,y]
resolPerResultants(pols, variables)

[[y == -1, x == 1], [y == 1, x == -1]]

In [18]:
pols = [
    x^2 - 1,
    y^2 - 4
]
variables = [x,y]
resolPerResultants(pols, variables)

[[y == -2, x == 1], [y == -2, x == -1], [y == 2, x == 1], [y == 2, x == -1]]

In [19]:
pols = [
    z^2 - 1,
    z - x,
    z - y
]
variables = [x,y,z]
resolPerResultants(pols, variables)

[[y == -1, x == -1, z == -1], [y == 1, x == 1, z == 1]]

In [20]:
pols = [
    x^2 - 1,
    y - x^2,
    z - x*y
]
variables = [x,y,z]
resolPerResultants(pols, variables)

[[z == -1, x == -1, y == 1], [z == 1, x == 1, y == 1]]

In [21]:
pols = [
    y - x^2,
    z - x^3
]
variables = [x,y,z]
resolPerResultants(pols, variables)

[[y == 1/2*(z^2)^(1/3)*(I*sqrt(3) - 1),
  x == -1/2*(z^2)^(2/3)*(I*sqrt(3) + 1)/z],
 [y == -1/2*(z^2)^(1/3)*(I*sqrt(3) + 1),
  x == 1/2*(z^2)^(2/3)*(I*sqrt(3) - 1)/z],
 [y == (z^2)^(1/3), x == (z^2)^(2/3)/z]]

In [22]:
pols = [
    x + y + z - 6,
    x - y,
    z - 2
]
variables = [x,y,z]
resolPerResultants(pols, variables)

[[y == 2, z == 2, x == 2]]

In [23]:
pols = [
    x + y + z + t - 10,
    x - y,
    y - z,
    z - t
]
variables = [x,y,z,t]
resolPerResultants(pols, variables)

[[t == (5/2), z == (5/2), y == (5/2), x == (5/2)]]

In [24]:
pols = [
    x^2 - 1,
    y - x,
    z - y^2,
    t - x*z
]
variables = [x,y,z,t]
resolPerResultants(pols, variables)

[[t == -1, y == -1, x == -1, z == 1], [t == 1, y == 1, x == 1, z == 1]]

In [25]:
pols = [
    x*y,
    x*z
]
variables = [x,y,z]
resolPerResultants(pols, variables)

[]

In [26]:
pols = [
    x^2 + 1,
    x^2 + 2
]
variables = [x]
resolPerResultants(pols, variables)

[]

In [27]:
pols = [
    x^2 - 1,
    y - x,
    z - y^2,
    t - z*x,
    u - t^2
]
variables = [x,y,z,t,u]
resolPerResultants(pols, variables)

[[u == 1, y == 1, t == 1, x == 1, z == 1],
 [u == 1, y == -1, t == -1, x == -1, z == 1]]